# **Ingest results.Json File**
1. Read all the files from the results folder using spark dataframe reader API
2. Define and enforce schema 
3. Add Metadata Columns
    .Source File
    .Ingestion Time Stamp 
3. Write to bronze delta table

In [0]:
%run ../00-common/01_Environment-Config

In [0]:
%run ../00-common/02_bronze_helpers

In [0]:
source_file = f"{landing_folder_path}/results"
table_name = f"{catalog_name}.{bronze_schema}.results"

In [0]:
source_file

#### **Step 1 Define and enforce schema (preserve nested structure)**

In [0]:
#Define the Nested Schema 
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType, DateType

results_schema = StructType(fields=[
    StructField("date", DateType(), True),    
    StructField("raceName", StringType(), True),
    StructField("round", IntegerType(), True),
    StructField("season", IntegerType(), True),    
    StructField("url", StringType(), True),
    StructField("constructorId", StringType(), True),
    StructField("driverId", StringType(), True),
    StructField("grid", IntegerType(), True),
    StructField("laps", IntegerType(), True),
    StructField("number", IntegerType(), True),
    StructField("points", FloatType(), True),
    StructField("position", IntegerType(), True),
    StructField("positionText", StringType(), True),
    StructField("status", StringType(), True)
])

#### **Step-2 Read the JSon file using the dataframe reader API**

In [0]:
results_df = (
    spark.read
    .format("json")
#   .option("inferSchema", True)
    .schema(results_schema)
    .option("header", True)
    .option('mode','FAILFAST')
    .load(source_file)
)

#### **Step-3. Add Metadata Columns**

In [0]:
results_final_df = add_ingestion_metadata(results_df)

In [0]:
display(results_final_df)

####  **Step 3. Write to bronze delta table**

In [0]:
(
    results_final_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)

In [0]:
%sql 
select 
season,
count(*) as season_Count
from 
formula1.bronze.results
group by season
order by season